# Body270 Regression — Starter Notebook

Goal: approximate the proprietary BMH05108 Body270 algorithm from (input, output) pairs collected
via `bmh05108-batch run`. Each row maps 15 input fields (demographics + 10 impedances) to ~65
output fields from the device.

**Workflow:**
1. Run `bmh05108-batch run` with a 100k-row input CSV and collect per-worker output CSVs.
2. Merge inputs + outputs on `row_id` (this notebook).
3. Inspect correlations and distribution of outputs.
4. Fit a regression model per output field.
5. Evaluate per-field R² and residuals.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

## 1. Load and merge CSVs

In [ ]:
DATA_DIR = Path("../data")

# Input samples (generated by generate_samples.py)
INPUT_CSV = DATA_DIR / "samples.csv"

# Output CSV(s) produced by bmh05108-batch run (may be per-worker, merge all)
OUTPUT_CSVS = sorted(DATA_DIR.glob("output_worker_*.csv"))

df_input = pd.read_csv(INPUT_CSV)
df_output = pd.concat([pd.read_csv(p) for p in OUTPUT_CSVS], ignore_index=True)

print(f"Input rows:  {len(df_input):,}")
print(f"Output rows: {len(df_output):,}")

df = df_input.merge(df_output, on="row_id", how="inner")
print(f"Merged rows: {len(df):,}")
df.head()

## 2. Inspect outputs

In [ ]:
# Only successful rows
df = df[df["status"] == "ok"].copy()
print(f"Valid rows: {len(df):,}")

# Summary stats for key outputs
TARGET_COLS = [
    "body_weight_kg", "body_fat_percent", "body_fat_mass_kg",
    "fat_free_mass_kg", "skeletal_muscle_mass_kg", "bmi",
    "basal_metabolism_kcal",
]
df[TARGET_COLS].describe().round(2)

## 3. Correlation matrix (inputs vs. key outputs)

In [ ]:
FEATURE_COLS = [
    "gender", "age", "height_cm", "weight_kg",
    "rh_20k", "lh_20k", "trunk_20k", "rf_20k", "lf_20k",
    "rh_100k", "lh_100k", "trunk_100k", "rf_100k", "lf_100k",
]

corr = df[FEATURE_COLS + TARGET_COLS].corr().loc[TARGET_COLS, FEATURE_COLS]
corr.round(3)

## 4. Linear regression baseline

In [ ]:
X = df[FEATURE_COLS].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

results = []

for target in TARGET_COLS:
    y = df[target].values
    X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

    model = LinearRegression()
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    results.append({
        "target": target,
        "r2": round(r2_score(y_te, y_pred), 4),
        "mae": round(mean_absolute_error(y_te, y_pred), 4),
    })

pd.DataFrame(results).sort_values("r2", ascending=False)

## 5. Next steps

- Add polynomial / interaction features for non-linear targets (body_fat_percent in particular)
- Fit per-segment outputs (right_hand_fat_percent, trunk_muscle_mass_kg, …)
- Evaluate residuals by age group and gender
- Try gradient boosting (XGBoost / LightGBM) as a stronger baseline before physics-informed regression